# AskQE Pipeline - NLLB Backtranslation QA

This notebook runs Question Answering using **NLLB backtranslations**.
- Uses pre-generated questions from the baseline QG
- Uses NLLB backtranslations from `nllb_qg_merged.jsonl`
- **Run each language cell individually** - you can start with Russian and continue later

**Input:** `nllb_qg_merged.jsonl`  
**Output:** `results Qwen3B baseline/biomqm/nllb/QA/bt-{lang}-vanilla.jsonl`

## 0. Detect Environment & Configure Cache

In [3]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f'Environment: {"Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    print(f'Model cache: {DRIVE_CACHE_DIR}')
elif IN_KAGGLE:
    KAGGLE_CACHE_DIR = '/kaggle/working/models_cache'
    os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = KAGGLE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(KAGGLE_CACHE_DIR, 'transformers')
    print(f'Model cache: {KAGGLE_CACHE_DIR}')

Environment: Kaggle
Model cache: /kaggle/working/models_cache


## 1. Setup - Install Dependencies & Clone Repository

In [4]:
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'accelerate', 'nltk', 'sentence-transformers', 'sacrebleu', 'textstat'], check=True)

if IN_COLAB:
    if not os.path.exists('/content/askqe'):
        subprocess.run(['git', 'clone', 'https://github.com/laurabon/AskQE_DNLP_2025-2026.git', '/content/askqe'], check=True)
    PROJECT_ROOT = '/content/askqe'
elif IN_KAGGLE:
    if not os.path.exists('/kaggle/working/askqe'):
        subprocess.run(['git', 'clone', 'https://github.com/laurabon/AskQE_DNLP_2025-2026.git', '/kaggle/working/askqe'], check=True)
    PROJECT_ROOT = '/kaggle/working/askqe'
else:
    # Find project root by looking for 'results Qwen3B baseline' or going up
    current_dir = os.getcwd()
    if 'results Qwen3B baseline' in current_dir:
        PROJECT_ROOT = current_dir.split('results Qwen3B baseline')[0].rstrip(os.sep)
    else:
        PROJECT_ROOT = current_dir

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.8 MB/s eta 0:00:00


Cloning into '/kaggle/working/askqe'...


Project root: /kaggle/working/askqe


Updating files: 100% (1422/1422), done.


## 2. Define Paths

In [5]:
MERGED_INPUT = os.path.join(RESULTS_DIR, 'backtranslation', 'nllb_qg_merged.jsonl')
QA_SCRIPT = os.path.join(RESULTS_DIR, 'biomqm', 'direct-prompting', 'code', 'qwen-3b-direct-prompting.py')
OUTPUT_DIR = os.path.join(RESULTS_DIR, 'biomqm', 'nllb', 'QA')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Input: {MERGED_INPUT}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'\n✓ Input exists: {os.path.exists(MERGED_INPUT)}')

Input: /kaggle/working/askqe/results Qwen3B baseline/backtranslation/nllb_qg_merged.jsonl
Output dir: /kaggle/working/askqe/results Qwen3B baseline/biomqm/nllb/QA

✓ Input exists: True


## 3. Pre-download Qwen Model

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('✓ Model cached')

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading Qwen/Qwen2.5-3B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Model cached


## 4. Verify Input Data

In [7]:
import json
from collections import Counter

with open(MERGED_INPUT, 'r', encoding='utf-8') as f:
    lines = f.readlines()

lang_counts = Counter(json.loads(line).get('lang_tgt', '?') for line in lines)
print(f'Total: {len(lines)} rows\n')
for lang, count in sorted(lang_counts.items()):
    print(f'  {lang}: {count}')

Total: 5216 rows

  de: 1309
  es: 801
  fr: 757
  ru: 680
  zh-CN: 1669


---

## 5. Question Answering

Run each language cell individually. Start with Russian, then continue with others later.

In [6]:
# SOURCE QA (English original)
print('=== Source QA ===')
output = os.path.join(OUTPUT_DIR, 'source-vanilla.jsonl')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'source', '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

=== Source QA ===


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda
Pipeline: direct-prompting
Generation params: temperature=0.1, top_p=0.9, repetition_penalty=1.1


`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-09 10:24:13.087758: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770632653.112125     177 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770632653.117773     177 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770632653.133168     177 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770632653.133190     177 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770632653.133193     177


=== SOURCE QA (Direct Prompting) ===
Using source prompt (no error warning)
Found 489 unique src values from 5216 total rows
[1/489] Processing src with 4 questions...
> First answer: Three cases are presented....
[2/489] Processing src with 8 questions...
> First answer: There were three patients in total....
[3/489] Processing src with 3 questions...
> First answer: Rapid progression and systemic illness are typical for the d...
[4/489] Processing src with 6 questions...
> First answer: NF was diagnosed based on clinical symptoms, laboratory para...
[5/489] Processing src with 6 questions...
> First answer: Necrotizing fasciitis (NF) is a life-threatening, usually ba...
[6/489] Processing src with 4 questions...
> First answer: Immediate surgical and antimicrobial therapy is required....
[7/489] Processing src with 5 questions...
> First answer: All patients showed rapidly progressing, painful swelling an...
[8/489] Processing src with 7 questions...
> First answer: NF refers to nec

### 🇷🇺 Russian (ru)

In [7]:
# RUSSIAN
lang = 'ru'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

=== Processing ru ===


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda
Pipeline: direct-prompting
Generation params: temperature=0.1, top_p=0.9, repetition_penalty=1.1


`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-09 11:20:49.240713: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770636049.260852     210 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770636049.266551     210 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770636049.281709     210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770636049.281730     210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770636049.281733     210


=== BT QA for ru (Direct Prompting) ===
Using bt prompt (with error warning)
Found 372 unique (src, bt_tgt) pairs from 680 rows for ru
[1/372] Processing bt for ru with 1 questions...
> First answer: The mental health risk factors for health workers are presen...
[2/372] Processing bt for ru with 4 questions...
> First answer: The article considers the impact of the COVID-19 pandemic on...
[3/372] Processing bt for ru with 2 questions...
> First answer: Psychotherapeutic and psychopharmaceutical approaches are pr...
[4/372] Processing bt for ru with 6 questions...
> First answer: This review presents and summarizes current literature on th...
[5/372] Processing bt for ru with 8 questions...
> First answer: Lithium...
[6/372] Processing bt for ru with 4 questions...
> First answer: stroke associated with...
[7/372] Processing bt for ru with 4 questions...
> First answer: Almost all post-COVID-19 patients complain of severe fatigue...
[8/372] Processing bt for ru with 4 questions...
> F

### 🇩🇪 German (de)

In [8]:
# GERMAN
lang = 'de'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

=== Processing de ===


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda
Pipeline: direct-prompting
Generation params: temperature=0.1, top_p=0.9, repetition_penalty=1.1


`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-09 15:54:30.080282: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770652470.100685     172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770652470.106163     172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770652470.120763     172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770652470.120785     172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770652470.120787     172


=== BT QA for de (Direct Prompting) ===
Using bt prompt (with error warning)
Found 614 unique (src, bt_tgt) pairs from 1309 rows for de
Resuming: 0 already processed
[1/614] Processing bt for de with 4 questions...
> First answer: Three cases are presented....
[2/614] Processing bt for de with 8 questions...
> First answer: There were three patients in total....
[3/614] Processing bt for de with 3 questions...
> First answer: Rapid progression and systemic disease are typical for the d...
[4/614] Processing bt for de with 6 questions...
> First answer: NF was diagnosed based on clinical symptoms, laboratory para...
[5/614] Processing bt for de with 6 questions...
> First answer: Necrotizing fasciitis (NF) is a severe, potentially fatal ba...
[6/614] Processing bt for de with 4 questions...
> First answer: Immediate surgical and antimicrobial therapy is required....
[7/614] Processing bt for de with 5 questions...
> First answer: All patients showed progressive rash, painful swelling a

### 🇪🇸 Spanish (es)

In [9]:
# SPANISH
lang = 'es'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

=== Processing es ===


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda
Pipeline: direct-prompting
Generation params: temperature=0.1, top_p=0.9, repetition_penalty=1.1


`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-09 17:16:25.802753: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770657385.823292     203 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770657385.829361     203 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770657385.845919     203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770657385.845943     203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770657385.845945     203


=== BT QA for es (Direct Prompting) ===
Using bt prompt (with error warning)
Found 495 unique (src, bt_tgt) pairs from 801 rows for es
[1/495] Processing bt for es with 3 questions...
> First answer: In the field of precision medicine....
[2/495] Processing bt for es with 7 questions...
> First answer: The review focused on available biomarkers, the role of incr...
[3/495] Processing bt for es with 6 questions...
> First answer: Advances in biomarkers have been made....
[4/495] Processing bt for es with 3 questions...
> First answer: Transurethral resection of the bladder...
[5/495] Processing bt for es with 4 questions...
> First answer: The molecular classification of bladder cancer represents on...
[6/495] Processing bt for es with 5 questions...
> First answer: The aim of the research was to explore the relationship betw...
[7/495] Processing bt for es with 5 questions...
> First answer: An association and cross-sectional study was performed....
[8/495] Processing bt for es with 6

### 🇫🇷 French (fr)

In [10]:
# FRENCH
lang = 'fr'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

=== Processing fr ===


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda
Pipeline: direct-prompting
Generation params: temperature=0.1, top_p=0.9, repetition_penalty=1.1


`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-09 18:22:14.653011: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770661334.672841     231 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770661334.678064     231 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770661334.692522     231 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770661334.692544     231 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770661334.692547     231


=== BT QA for fr (Direct Prompting) ===
Using bt prompt (with error warning)
Found 347 unique (src, bt_tgt) pairs from 757 rows for fr
[1/347] Processing bt for fr with 3 questions...
> First answer: Military deficiency in adulthood is being discussed, but the...
[2/347] Processing bt for fr with 3 questions...
> First answer: Hemoglobin participates in various metabolisms....
[3/347] Processing bt for fr with 6 questions...
> First answer: LAID is caused by bleeding, usually from the gastrointestina...
[4/347] Processing bt for fr with 3 questions...
> First answer: C-reactive protein levels are high....
[5/347] Processing bt for fr with 2 questions...
> First answer: Iron is a major mineral in the human body....
[6/347] Processing bt for fr with 9 questions...
> First answer: The two types of ID are Absolute ID (IDA) and Functional ID ...
[7/347] Processing bt for fr with 5 questions...
> First answer: AID is not explicitly defined in the provided context. It ap...
[8/347] Processin

### 🇨🇳 Chinese (zh-CN)

In [11]:
# CHINESE
lang = 'zh-CN'
output = os.path.join(OUTPUT_DIR, f'bt-{lang}-vanilla.jsonl')
print(f'=== Processing {lang} ===')
subprocess.run([sys.executable, '-u', QA_SCRIPT, '--mode', 'bt', '--lang', lang, '--qg_input_path', MERGED_INPUT, '--output_path', output], check=True)
print(f'✓ Done: {output}')

=== Processing zh-CN ===


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda
Pipeline: direct-prompting
Generation params: temperature=0.1, top_p=0.9, repetition_penalty=1.1


`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-09 19:05:39.252605: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770663939.272942     259 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770663939.278722     259 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770663939.293801     259 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770663939.293822     259 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770663939.293825     259


=== BT QA for zh-CN (Direct Prompting) ===
Using bt prompt (with error warning)
Found 1003 unique (src, bt_tgt) pairs from 1669 rows for zh-CN
[1/1003] Processing bt for zh-CN with 4 questions...
> First answer: The context does not specify what exact drugs are being refe...
[2/1003] Processing bt for zh-CN with 4 questions...
> First answer: The most feared complication is inflammation of the eye (a s...
[3/1003] Processing bt for zh-CN with 4 questions...
> First answer: Developing a continuous delivery platform for VEGF drugs to ...
[4/1003] Processing bt for zh-CN with 3 questions...
> First answer: Various strategies have been conceptualized in recent years....
[5/1003] Processing bt for zh-CN with 2 questions...
> First answer: The characteristics of the ideal continuous delivery platfor...
[6/1003] Processing bt for zh-CN with 4 questions...
> First answer: This review aims to provide an overview of continuous delive...
[7/1003] Processing bt for zh-CN with 2 questions...
> Fir

---

## 6. Check Results

In [12]:
print('=== Output Files ===')
if os.path.exists(OUTPUT_DIR):
    for f in sorted(os.listdir(OUTPUT_DIR)):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024*1024)
        print(f'  {f}: {size:.2f} MB')

=== Output Files ===
  bt-de-vanilla.jsonl: 0.63 MB
  bt-es-vanilla.jsonl: 0.58 MB
  bt-fr-vanilla.jsonl: 0.34 MB
  bt-zh-CN-vanilla.jsonl: 1.22 MB


## 7. Download (Kaggle)

In [ ]:
if IN_KAGGLE:
    import shutil
    shutil.make_archive('/kaggle/working/nllb_qa_results', 'zip', OUTPUT_DIR)
    print('✓ Results zipped - download from Output tab')

---

# Integrated Steps from Baseline (3, 4, 5)


## 3. BioMQM Pipeline

In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, 'biomqm', 'askqe'))
output_path = os.path.join(RESULTS_DIR, 'biomqm', 'askqe_qg_qwen3b.jsonl')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
subprocess.run([sys.executable, '-u', 'qwen-3b.py', '--output_path', output_path, '--prompt', 'atomic'], check=True)

---

## 4. Evaluation Metrics

### 4.1 SBERT

In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, 'evaluation', 'sbert'))
output_file = os.path.join(RESULTS_DIR, 'evaluation', 'sbert', 'qwen-3b.csv')
os.makedirs(os.path.dirname(output_file), exist_ok=True)
subprocess.run([sys.executable, 'sbert.py', '--model', 'qwen-3b', '--output_file', output_file], check=True)

### 4.2 String Comparison

In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, 'evaluation', 'string-comparison'))
subprocess.run([sys.executable, 'string_comparison.py'], check=True)

### 4.3 BT-Score

In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, 'evaluation', 'bt-score'))
subprocess.run([sys.executable, 'run_bt.py'], check=True)

---

## 5. Desiderata Evaluation

In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, 'evaluation', 'desiderata'))
subprocess.run([sys.executable, 'i_avg_questions.py'], check=True)
subprocess.run([sys.executable, 'i_duplicate.py'], check=True)
subprocess.run([sys.executable, 'i_diversity.py'], check=True)
subprocess.run([sys.executable, 'q_answerability.py'], check=True)
subprocess.run([sys.executable, 'q_readability.py'], check=True)